# 08 — Decorators with Function Arguments

This notebook focuses on **arguments passed to the decorated function**. It builds on basic decorators, decorator factories, and `functools.wraps` without reteaching them.

The key goal is to make decorators reusable for functions that accept different positional and keyword arguments while preserving their return values.


## 1. Introduction

There are two different kinds of arguments that can appear around a decorator:

- **Decorator arguments** — configuration passed to a decorator factory, such as `@repeat(3)`.
- **Function arguments** — values passed to the decorated function, such as `greet("Komal")`.

In this notebook, the focus is the second kind. A decorator must receive those arguments and forward them to the original function.


In [ ]:
@decorator
def greet(name):
    print(f"Hello, {name}")

greet("Komal")


Here, `name` is an argument of `greet()`. By contrast, in `@repeat(3)`, the `3` is an argument to the decorator factory. Keeping these two ideas separate prevents a lot of decorator confusion.

## 2. Review: Basic Decorators

A basic decorator receives a function, creates a wrapper, and returns the wrapper. This is only a short review because the decorator mechanism was covered earlier.

In [ ]:
def decorator(func):
    def wrapper():
        print("Before")
        func()
        print("After")
    return wrapper

@decorator
def greet():
    print("Hello")

greet()


This works for a function with no parameters. The next problem is what happens when the decorated function needs arguments.

## 3. The Problem with Function Arguments

Suppose the same decorator is applied to a function that accepts `name`.

In [ ]:
def decorator(func):
    def wrapper():
        print("Before")
        func()
        print("After")
    return wrapper

@decorator
def greet(name):
    print(f"Hello, {name}")

# greet("Komal")
# The call would fail because wrapper() accepts no arguments.


The decorated name now refers to `wrapper`. Calling `greet("Komal")` effectively tries to call `wrapper("Komal")`, but `wrapper()` has no parameter. The wrapper therefore has to accept and forward the function's arguments.

## 4. Decorator Wrapper with Fixed Arguments

If we know the decorated function always has one parameter called `name`, we can write a matching wrapper.

In [ ]:
def decorator(func):
    def wrapper(name):
        print("Before")
        func(name)
        print("After")
    return wrapper

@decorator
def greet(name):
    print(f"Hello, {name}")

greet("Komal")


The flow is:

```text
greet("Komal")
    ↓
wrapper("Komal")
    ↓
func("Komal")
```

This works, but it is tied to one specific function signature. A reusable decorator should work with many different signatures.

## 5. Passing Arguments to the Original Function

A decorator should normally preserve the original function's result. Capture the result, perform any post-processing, and return it.

In [ ]:
def decorator(func):
    def wrapper(number):
        print("Before")
        result = func(number)
        print("After")
        return result
    return wrapper

@decorator
def square(number):
    return number ** 2

result = square(5)
print(result)


The important pattern is:

```python
result = func(...)
return result
```

Without the `return`, the decorated function would return `None` even though the original function returned a value.

## 6. Using `*args` and `**kwargs`

`*args` and `**kwargs` were already covered in `01_Functions_and_Scope/03_Args_and_Kwargs.ipynb`. We will use them here without reteaching their mechanics.

They let a wrapper accept arbitrary positional and keyword arguments and forward them to the original function.

In [ ]:
from functools import wraps

def decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("Before")
        result = func(*args, **kwargs)
        print("After")
        return result
    return wrapper


In [ ]:
@decorator
def greet(name):
    print(f"Hello, {name}")

@decorator
def add(a, b):
    return a + b

@decorator
def introduce(name, age):
    return f"{name} is {age} years old."

greet("Komal")
print(add(10, 20))
print(introduce("Komal", 25))


The central forwarding pattern is:

```text
wrapper(*args, **kwargs)
        ↓
func(*args, **kwargs)
```

This makes the decorator reusable across functions with different signatures.

## 7. Preserving Return Values

Compare these two wrappers.

In [ ]:
def bad_decorator(func):
    def wrapper(*args, **kwargs):
        func(*args, **kwargs)
    return wrapper

@bad_decorator
def add(a, b):
    return a + b

print(add(10, 20))  # None


In [ ]:
def good_decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return result
    return wrapper

@good_decorator
def add(a, b):
    return a + b

print(add(10, 20))  # 30


A decorator can add behavior before or after a function call, but it should not accidentally discard the original return value.

## 8. Decorators That Modify Arguments

A decorator can inspect or transform an argument before passing it to the original function.

In [ ]:
def uppercase_name(func):
    @wraps(func)
    def wrapper(name):
        name = name.upper()
        return func(name)
    return wrapper

@uppercase_name
def greet(name):
    return f"Hello, {name}"

print(greet("komal"))


The flow is:

```text
original argument
      ↓
decorator modifies it
      ↓
modified argument
      ↓
original function
```

Argument transformation should be intentional because it changes what the original function receives.

## 9. Decorators That Validate Arguments

Validation is another useful decorator pattern. The wrapper checks an argument before allowing the original function to run.

In [ ]:
def require_positive(func):
    @wraps(func)
    def wrapper(number):
        if number <= 0:
            raise ValueError("Number must be positive")
        return func(number)
    return wrapper

@require_positive
def square(number):
    return number ** 2

print(square(5))


In [ ]:
try:
    square(-2)
except ValueError as error:
    print(error)


For decorators that need to support many function signatures, `*args` and `**kwargs` can be used to inspect or validate the forwarded arguments without introducing a new argument-handling lesson.

In [ ]:
def log_positive_call(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        if args and args[0] <= 0:
            raise ValueError("First positional argument must be positive")
        return func(*args, **kwargs)
    return wrapper


## 10. Decorators That Log Function Calls

A logging decorator can record the function name, arguments, and result while leaving the function's main behavior unchanged.

In [ ]:
def log_call(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}")
        print(f"Arguments: {args}")
        print(f"Keyword arguments: {kwargs}")

        result = func(*args, **kwargs)

        print(f"Result: {result}")
        return result
    return wrapper

@log_call
def add(a, b):
    return a + b

add(10, 20)


The call flow is:

```text
add(10, 20)
    ↓
wrapper(10, 20)
    ↓
logging
    ↓
original add(10, 20)
    ↓
result
    ↓
return result
```

## 11. Decorators That Measure Function Execution

A timing-style decorator can measure how long the original function takes. The important lesson here is the transparent wrapper pattern, not performance optimization.

In [ ]:
import time

def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()

        result = func(*args, **kwargs)

        end = time.perf_counter()
        print(f"{func.__name__} took {end - start:.6f} seconds")
        return result
    return wrapper


In [ ]:
@timer
def calculate():
    total = 0
    for number in range(1_000_000):
        total += number
    return total

print(calculate())


The decorator surrounds a function call, receives its arguments, runs it, measures the surrounding work, and returns its result.

## 12. Decorators with Positional and Keyword Arguments

A reusable wrapper should support both positional and keyword calls.

In [ ]:
@log_call
def introduce(name, age, city=None):
    return f"{name}, {age}, {city}"

print(introduce("Komal", 25))
print(introduce("Komal", 25, "Delhi"))
print(introduce(name="Komal", age=25, city="Delhi"))


All three calls can be forwarded by the same wrapper:

```text
wrapper(*args, **kwargs)
        ↓
func(*args, **kwargs)
```

This is what makes the decorator reusable across functions with different combinations of positional and keyword arguments.

## 13. Practical Decorator Examples

### Example 1 — Logging

In [ ]:
@log_call
def multiply(a, b):
    return a * b

multiply(4, 5)


### Example 2 — Validation

In [ ]:
@require_positive
def calculate_square(number):
    return number ** 2

print(calculate_square(6))


### Example 3 — Argument Transformation

In [ ]:
def strip_text(func):
    @wraps(func)
    def wrapper(text):
        return func(text.strip())
    return wrapper

@strip_text
def display(text):
    return f"Text: {text}"

print(display("   Hello Python   "))


### Example 4 — Combining Decorators

This reinforces decorator stacking from `05_Basic_Decorators.ipynb` without reteaching it in depth.

In [ ]:
@timer
@log_call
def calculate(a, b):
    return a + b

print(calculate(10, 20))


## 14. Common Mistakes

### Mistake 1 — The wrapper does not accept arguments

```python
def wrapper():
    ...
```

This fails when the decorated function receives arguments.

### Mistake 2 — Arguments are not forwarded

```python
func()
```

instead of:

```python
func(*args, **kwargs)
```

### Mistake 3 — The return value is lost

```python
func(*args, **kwargs)
```

instead of:

```python
return func(*args, **kwargs)
```

### Mistake 4 — Confusing decorator arguments with function arguments

In `@repeat(3)` the `3` configures the decorator. In `greet(name)`, `name` belongs to the decorated function.

### Mistake 5 — Forgetting `@wraps`

`functools.wraps` was covered in `07_Functools_Wraps.ipynb`. Use it when writing reusable decorators so important function metadata is preserved.

## 15. Summary

The reusable mental model is:

```text
Original Function
      │
      │ accepts arguments
      ▼
   Decorator
      │
      ▼
    Wrapper
      │
      │ receives *args, **kwargs
      ▼
 Pre-processing
      │
      ▼
Original Function
      │
      │ returns result
      ▼
 Post-processing
      │
      ▼
  return result
```

The central reusable pattern is:

```python
from functools import wraps

def decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        # before
        result = func(*args, **kwargs)
        # after
        return result
    return wrapper
```

### Boundary

This notebook does not introduce new material on `*args`, `**kwargs`, `functools.wraps`, decorator arguments, closures, or basic decorator syntax. Those topics were covered earlier. The specific goal here is making decorators work correctly with **functions that receive arguments and return values**.

**Next:** `09_Class_Decorators.ipynb`